# Held-out log-likelihood of the conditional NF

Does the flow generalise, or has it just memorised the frames it was fitted on?

`scripts/train_conditional_nf.py` optimises `-log_prob` over pixels of the **train**
split only, so its loss curve says nothing about generalization. This notebook scores the
**same objective** -- `log_prob` of (frozen-NeRF surface point, ray direction) conditioned
on that pixel's DINOv2 feature -- on the train frames *and* on nerfstudio's held-out
frames (~10% at the 0.9 `train_split_fraction` default), and reports the gap. A large
positive gap means the flow fits what it saw far better than what it did not.

It reuses the trainer's own `build_training_cameras`, `precompute_dino_cache` and
`sample_batch`, so what is measured here IS the training objective: same continuous
sub-pixel sampling, same DINO patch lookup, same frozen-NeRF depth render.

### Read the median, not the mean

`log_prob` is a log-**density** of a continuous 6-D distribution, so it is unbounded in
both directions. The target also lives on a lower-dimensional manifold -- directions are
unit vectors, and `points = origins + directions * depth` are NeRF surface points -- so
the flow pushes the density very high on that manifold and off a cliff just outside it.
One ray whose frozen-NeRF depth is garbage (a window, the unbounded background) lands far
off it and gets a catastrophically negative value.

In an earlier run on `counter`, **one sample out of 81,920** at `log_prob ≈ -13,400`
produced `std_log_prob = 47.5` around a mean of `9.67`, and pulled that mean down by 0.17
on its own. The mean and std are real numbers, but they describe that one sample, not the
distribution.

So the report now carries, alongside the untouched `mean_log_prob` / `std_log_prob` /
`train_minus_test`:

- `median_log_prob`, `trimmed_mean_log_prob_0.1pct`, `iqr_log_prob`, `mad_log_prob`
  -- the centre and spread, undistorted;
- `quantiles` from p0 to p100, so you can see where the tail starts;
- `tail_contribution` -- for the lowest 0.001 / 0.01 / 0.1 / 1%, how much of the mean and
  of the variance they carry. No inference required;
- `worst_samples` -- the blow-ups themselves with their rendered **depth**, `|P|` and
  source image, which is what tests the bad-depth explanation;
- `spearman_log_prob_vs_depth`, `per_image`, `batch_mins` / `batch_maxes`, a `histogram`,
  and a `<scene>_cond_nf_ll_samples.npz` sidecar with every raw value for re-analysis.

**This notebook never trains.** It restores a nerfacto backbone and a conditional-NF
checkpoint from an attached Dataset and fails loudly if a scene has neither -- train them
with `notebooks/train_vf_nerf_kaggle.ipynb` first, then attach that run's
`vf_nerf_outputs.zip` as-is. The flow and the backbone must come from the same run: the
flow models points in *that* NeRF's normalised world frame.

Worker script: `scripts/eval_cond_nf_likelihood.py`.

In [ ]:
# @title 0. Environment sanity check  (STOP if this cell raises)
import sys, platform, subprocess, os
print('system python:', sys.version)
print('platform:', platform.platform())
smi = subprocess.run(['bash','-c','nvidia-smi -L 2>/dev/null || true'], capture_output=True, text=True).stdout.strip()
print('GPU:', smi or '(none)')
if not smi:
    raise RuntimeError(
        'No GPU. Right sidebar -> Settings -> Accelerator -> GPU T4 x2 (or P100), '
        'and Internet -> On (needs a phone-verified account). Then re-run.')
net = subprocess.run(['bash','-c','curl -sI --max-time 10 https://pypi.org >/dev/null && echo ok || echo fail'], capture_output=True, text=True).stdout.strip()
print('internet:', net)
if net != 'ok':
    raise RuntimeError('No internet. Right sidebar -> Settings -> Internet -> On, then re-run.')
for d in ('/kaggle/temp', '/kaggle/working'):
    os.makedirs(d, exist_ok=True)


# Scenes to train + probe in this run (all Mip-NeRF 360, indoor/bounded -- see
# scripts/downloads/download_mipnerf360.py). Trim this list to train fewer.
SCENES = ['counter', 'kitchen', 'room']
_ALLOWED_SCENES = ('bonsai', 'counter', 'kitchen', 'room')  # == downloader's --scene choices
assert SCENES and all(s in _ALLOWED_SCENES for s in SCENES), f'bad SCENES {SCENES}'
print('scenes:', SCENES)

In [ ]:
# @title 1. Build isolated CUDA 11.7 / Python 3.10 env + tiny-cuda-nn + repo
# Mirrors the debugged Colab setup: torch==1.13.1+cu117 has no cp311 wheel and
# tiny-cuda-nn needs the CUDA 11.7 dev headers, so we build a py3.10 venv rather
# than touch Kaggle's system interpreter.
setup = r'''#!/bin/bash
set -e
export DEBIAN_FRONTEND=noninteractive

TMP=/kaggle/temp
REPO=$TMP/VF-NeRF-conditional
VENV=$TMP/venv310
mkdir -p $TMP

echo '=== add NVIDIA CUDA apt repo (for the 11.7 dev packages) ==='
UBU=$(. /etc/os-release && echo ${VERSION_ID//./})   # 2204 / 2404
if ! ls /etc/apt/sources.list.d/ | grep -qi cuda; then
  wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu${UBU}/x86_64/cuda-keyring_1.1-1_all.deb || \
  wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
  dpkg -i /tmp/cuda-keyring.deb
fi

echo '=== apt packages ==='
apt-get -qq update
# python3.10: default on 22.04; via deadsnakes otherwise
if ! apt-get -qq install -y python3.10 python3.10-venv python3.10-dev 2>/dev/null; then
  apt-get -qq install -y software-properties-common
  add-apt-repository -y ppa:deadsnakes/ppa
  apt-get -qq update
  apt-get -qq install -y python3.10 python3.10-venv python3.10-dev
fi
apt-get -q install -y \
    cuda-nvcc-11-7 cuda-cudart-dev-11-7 cuda-nvrtc-dev-11-7 libcublas-dev-11-7 libcufft-dev-11-7 \
    libcurand-dev-11-7 libcusolver-dev-11-7 libcusparse-dev-11-7 libnpp-dev-11-7 \
    libnvjpeg-dev-11-7 ninja-build ffmpeg

if [ ! -x /usr/local/cuda-11.7/bin/nvcc ]; then
  echo 'FATAL: /usr/local/cuda-11.7/bin/nvcc missing after apt install'
  ls -R /usr/local/cuda-11.7 2>/dev/null | head -40; apt-cache policy cuda-nvcc-11-7
  exit 1
fi

echo '=== register CUDA 11.7 lib path ==='
echo '/usr/local/cuda-11.7/lib64' > /etc/ld.so.conf.d/cuda-11-7.conf && ldconfig
# tiny-cuda-nn links against the CUDA driver lib (-lcuda). apt CUDA has only the
# stub; the real one ships with the GPU driver as libcuda.so.1 with no .so symlink.
STUB=/usr/local/cuda-11.7/lib64/stubs
REAL=$(ldconfig -p | awk '/libcuda\.so\.1/{print $NF; exit}')
[ -n "$REAL" ] && [ ! -e /usr/local/cuda-11.7/lib64/libcuda.so ] && ln -sf "$REAL" /usr/local/cuda-11.7/lib64/libcuda.so
export LIBRARY_PATH=/usr/local/cuda-11.7/lib64:$STUB${LIBRARY_PATH:+:$LIBRARY_PATH}
echo "libcuda: real=$REAL  stub=$(ls $STUB/libcuda.so 2>/dev/null)"

echo '=== venv310 ==='
[ -d $VENV ] || python3.10 -m venv $VENV
$VENV/bin/pip install -q 'setuptools<81' wheel

echo '=== torch 1.13.1+cu117 ==='
$VENV/bin/pip install -q torch==1.13.1 torchvision functorch --extra-index-url https://download.pytorch.org/whl/cu117
$VENV/bin/pip install -q ninja

echo '=== toolchain diagnostics ==='
export CUDA_HOME=/usr/local/cuda-11.7
export PATH=/usr/local/cuda-11.7/bin:$PATH
ls -d /usr/local/cuda* || true
which nvcc; nvcc --version 2>&1 | tail -2 || echo 'NO nvcc at /usr/local/cuda-11.7/bin'
gcc --version | head -1; g++ --version | head -1
# CUDA 11.7 nvcc rejects gcc>11; pin to gcc-11 if a newer default is present
if gcc -dumpversion | grep -qvE '^(9|10|11)'; then
  apt-get -qq install -y gcc-11 g++-11
  export CC=gcc-11 CXX=g++-11
  export NVCC_PREPEND_FLAGS='-ccbin g++-11'
  echo 'pinned to gcc-11'
fi

echo '=== build tiny-cuda-nn ==='
# derive the arch from the actual GPU (T4 -> 75, P100 -> 60, L4 -> 89, ...)
GPUCC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '. ')
export TCNN_CUDA_ARCHITECTURES=${GPUCC:-75}
echo "TCNN_CUDA_ARCHITECTURES=$TCNN_CUDA_ARCHITECTURES"
# master built fine on Colab recently; fall back through the last few tags if it
# has since moved past CUDA 11.7. Override the whole list with TCNN_REFS.
TCNN_OK=
for ref in ${TCNN_REFS:-master v1.6 v1.5}; do
  [ "$ref" = master ] && spec='git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch' \
                       || spec="git+https://github.com/NVlabs/tiny-cuda-nn/@${ref}#subdirectory=bindings/torch"
  echo "--- trying tiny-cuda-nn @ $ref ---" | tee -a /kaggle/working/tcnn_build.log
  if $VENV/bin/pip install --no-build-isolation -v "$spec" >> /kaggle/working/tcnn_build.log 2>&1; then
    TCNN_OK=$ref; break
  fi
  echo "  @ $ref failed:"; grep -E 'error:|fatal error|unsupported|cannot find -l|undefined reference|ld returned' /kaggle/working/tcnn_build.log | tail -8
done
if [ -z "$TCNN_OK" ]; then
  echo '!!! tiny-cuda-nn build FAILED for every ref — full log at /kaggle/working/tcnn_build.log'
  grep -nE 'error:|fatal error|unsupported|cannot find -l|undefined reference|ld returned' /kaggle/working/tcnn_build.log | tail -40
  exit 1
fi
echo "tiny-cuda-nn OK (@ $TCNN_OK)"
$VENV/bin/python -c 'import tinycudann as t; print("tcnn import ok", t.__version__ if hasattr(t,"__version__") else "")'

echo '=== clone repo ==='
rm -rf $REPO
git clone --quiet https://github.com/itayhanoch/VF-NeRF-conditional.git $REPO

echo '=== apply known repo fixes ==='
# 1) eval call site missing the `step` arg (crashes at the first full-image eval)
sed -i 's/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch)/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch, step)/' \
    $REPO/nerfstudio/pipelines/base_pipeline.py
# 2) scripts/render.py imports get_mask_from_view_likelihood, removed with the
#    registration pipeline — dead import, never used. Strip it so ns-render works.
sed -i '/get_mask_from_view_likelihood/d' $REPO/scripts/render.py
# 3) DINOv2 hub code (facebookresearch/dinov2 @ main) now needs torch>=2.0
#    (F.scaled_dot_product_attention). Prepend a math-identical fallback to the
#    one module that torch.hub.load's DINOv2 — nerfstudio/utils/dino_features.py.
$VENV/bin/python - "$REPO/nerfstudio/utils/dino_features.py" <<'PYEOF'
import sys, pathlib
p = pathlib.Path(sys.argv[1])
s = p.read_text()
if 'sdpa-shim' not in s:
    shim = ('\n# sdpa-shim (DINOv2 main needs torch>=2.0; this stack is torch 1.13)\n'
            'import math as _m\n'
            'import torch as _t\n'
            'import torch.nn.functional as _F\n'
            'if not hasattr(_F, "scaled_dot_product_attention"):\n'
            '    def _sdpa(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):\n'
            '        sc = 1.0 / _m.sqrt(q.size(-1)) if scale is None else scale\n'
            '        a = _t.matmul(q, k.transpose(-2, -1)) * sc\n'
            '        if attn_mask is not None:\n'
            '            a = a.masked_fill(~attn_mask, float("-inf")) if attn_mask.dtype == _t.bool else a + attn_mask\n'
            '        a = a.softmax(-1)\n'
            '        if dropout_p:\n'
            '            a = _F.dropout(a, dropout_p)\n'
            '        return _t.matmul(a, v)\n'
            '    _F.scaled_dot_product_attention = _sdpa\n')
    # insert AFTER `from __future__` (must stay first statement); else after the docstring
    anchor = 'from __future__ import annotations\n'
    if anchor in s:
        s = s.replace(anchor, anchor + shim, 1)
    else:
        s = shim.lstrip() + '\n' + s
    p.write_text(s)
    print('dino_features.py: sdpa shim installed')
else:
    print('dino_features.py: sdpa shim already present')
PYEOF

echo '=== install repo + normalizing-flows ==='
cd $REPO
$VENV/bin/pip install -q --no-build-isolation -e . -e ./normalizing-flows

echo '=== SETUP COMPLETE ==='
'''
import os, subprocess
os.makedirs('/kaggle/temp', exist_ok=True)
with open('/kaggle/temp/setup.sh', 'w') as fh:
    fh.write(setup)
# pipefail so the cell sees bash's exit code, not tee's
rc = subprocess.call(
    ['bash', '-c', 'set -o pipefail; bash /kaggle/temp/setup.sh 2>&1 | tee /kaggle/working/setup.log'])
if rc != 0:
    print('--- last 60 lines of setup.log ---')
    print(subprocess.run(['tail', '-n', '60', '/kaggle/working/setup.log'], capture_output=True, text=True).stdout)
    raise RuntimeError(f'setup.sh failed (exit {rc}). See /kaggle/working/setup.log '
                       'and, for tiny-cuda-nn, /kaggle/working/tcnn_build.log')

In [ ]:
# @title 2. Config
import os, glob
TMP = '/kaggle/temp'
REPO = f'{TMP}/VF-NeRF-conditional'
VENV = f'{TMP}/venv310/bin'
WORK = '/kaggle/working'

# Per-scene paths (SCENES comes from cell 0).
SCENE_DATA = lambda s: f'{REPO}/data/mipnerf360/{s}'
COND_DIR   = lambda s: f'{WORK}/checkpoints/conditional_nf/{s}'

# nerfacto recipe mirrored from the original VF-NeRF (leosegre/VF_NeRF,
# reg_pipeline_pc.py): downscale 2, 60k iters, 1024 rays/batch, camera-opt off,
# full train split, center-method=focus, scene-scale 2. Applies to every scene.
NERFACTO_DOWNSCALE   = 2
NERFACTO_ITERS       = 60000
NERFACTO_RAYS        = 1024
NERFACTO_FORCE_RETRAIN = False   # True = ignore/delete any existing nerfacto run and retrain
COND_NF_MAX_STEPS    = 30000
COND_NF_FORCE_RETRAIN = True  # True = train even if a checkpoint is available for a scene
# --- conditional-NF architecture (a restored .pt keeps ITS OWN architecture --
# these only take effect on a scene that actually trains, hence FORCE_RETRAIN
# above when you change them) ---
COND_NF_NUM_BLOCKS = 8  # coupling (+batchnorm) blocks in the flow
COND_NF_HIDDEN_DIM = 128  # hidden width of every coupling layer's scale/translate MLPs
COND_NF_REDUCE_DIM = None  # reduce the DINO condition to this many dims via a jointly-trained MLP before conditioning the flow (None = no reduction; the flow sees the raw 384-d DINO feature)
COND_NF_REDUCE_DIVIDE_FACTOR = 8  # only used when COND_NF_REDUCE_DIM is set
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.7/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

NERF_OUTPUT_DIR = f'{WORK}/outputs'          # persisted; nerfacto nests <scene>/nerfacto/<ts>/
RENDER_DIR = f'{WORK}/renders'               # persisted
os.makedirs(RENDER_DIR, exist_ok=True)

# ---- per-scene checkpoint reuse: match by OUTPUT directory structure ----
# A scene is skipped (not retrained) if a matching checkpoint is found. Matching
# is structural -- the scene name must sit exactly where cell 5/7 themselves put
# it, mirroring the two layouts that actually occur (not a loose "scene name
# anywhere in the path" guess, which could mis-fire on e.g. a Dataset folder
# named "bathroom-photos"):
#   * nerfacto:       .../<scene>/nerfacto/<timestamp>/config.yml   (the raw
#                     nerfstudio output tree cell 5 trains into -- also what a
#                     *.tar.gz of outputs/ unpacks to)
#                     OR .../nerfacto_<scene>/config.yml             (cell 7's
#                     packaged bundle layout; also matches the repo's own
#                     checkpoints/nerfacto_<scene>/, so bonsai always resolves)
#   * conditional-NF: .../conditional_nf/<scene>/latest.pt (or cond_nf_step_*.pt)
#                     OR .../conditional_nf_<scene>/latest.pt (packaged bundle)
# Attach a Dataset (right sidebar -> Input -> Add Input) preserving one of these
# layouts intact for each scene you want to skip -- e.g. re-attach a prior run's
# vf_nerf_outputs.zip as-is (it already uses the nerfacto_<scene>/ and
# conditional_nf_<scene>/ bundle layout), or upload outputs/<scene>/ intact.
from pathlib import Path

def _nerf_scene_of(cfg_path):
    parts = Path(cfg_path).parts
    for i, seg in enumerate(parts):
        if seg == 'nerfacto' and i > 0 and parts[i - 1] in SCENES:
            return parts[i - 1]                       # .../<scene>/nerfacto/<ts>/config.yml
        if seg.startswith('nerfacto_') and seg[len('nerfacto_'):] in SCENES:
            return seg[len('nerfacto_'):]              # .../nerfacto_<scene>/config.yml
    return None

def _cond_scene_of(pt_path):
    parts = Path(pt_path).parts
    for i, seg in enumerate(parts):
        if seg == 'conditional_nf' and i + 1 < len(parts) and parts[i + 1] in SCENES:
            return parts[i + 1]                        # .../conditional_nf/<scene>/latest.pt
        if seg.startswith('conditional_nf_') and seg[len('conditional_nf_'):] in SCENES:
            return seg[len('conditional_nf_'):]         # .../conditional_nf_<scene>/latest.pt
    return None

PRETRAINED_NERF = {}            # scene -> config.yml (any attached Dataset OR the repo's own checkpoints/)
for c in (glob.glob('/kaggle/input/**/config.yml', recursive=True) +
          glob.glob(f'{REPO}/checkpoints/**/config.yml', recursive=True)):
    # ckpt may sit under nerfstudio_models/ (the usual nerfstudio layout, kept
    # intact by cell 7's packaging) or flat next to config.yml (the repo's own
    # committed checkpoints/nerfacto_bonsai/) -- accept either.
    if glob.glob(os.path.join(os.path.dirname(c), '**', '*.ckpt'), recursive=True):
        s = _nerf_scene_of(c)
        if s:
            PRETRAINED_NERF.setdefault(s, c)

INPUT_COND_PT = {}              # scene -> latest.pt / cond_nf_step_*.pt
for p in sorted(glob.glob('/kaggle/input/**/latest.pt', recursive=True) +
                glob.glob('/kaggle/input/**/cond_nf_step_*.pt', recursive=True)):
    s = _cond_scene_of(p)
    if s:
        INPUT_COND_PT.setdefault(s, p)

PRETRAINED_TARS = sorted(glob.glob('/kaggle/input/**/*.tar.gz', recursive=True) +
                         glob.glob('/kaggle/input/**/*.tgz', recursive=True))  # scene(s) inferred on extract, same structural rule

print('nerfacto restore :', PRETRAINED_NERF)
print('nerfacto tarballs:', PRETRAINED_TARS)
print('cond-NF restore  :', INPUT_COND_PT)

In [ ]:
# @title 2b. Download scenes + build downscaled images
import os, subprocess
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

for s in SCENES:
    dd = SCENE_DATA(s)
    if not os.path.isdir(f'{dd}/images'):
        subprocess.run([f'{VENV}/python', 'scripts/downloads/download_mipnerf360.py',
                        '--scene', s, '--save-dir', f'{REPO}/data/mipnerf360'],
                       check=True, cwd=REPO)
    else:
        print(f'{dd}/images present, skip download')

    # nerfstudio 0.2.1 does NOT auto-generate images_<N>/; the dataparser just
    # looks for it. downscale 2 = training res (VF-NeRF recipe), 4 = spare.
    # Full res OOMs the image cache on a free-tier box.
    src = Path(dd) / 'images'
    n_src = len(list(src.glob('*')))
    for factor in sorted({NERFACTO_DOWNSCALE, 4}):
        dst = Path(dd) / f'images_{factor}'
        if dst.is_dir() and len(list(dst.glob('*'))) == n_src:
            print(f'  {s}/images_{factor} present, skip')
            continue
        dst.mkdir(exist_ok=True)
        def _f(p, factor=factor, dst=dst):
            im = Image.open(p); w, h = im.size
            im.resize((w // factor, h // factor), Image.LANCZOS).save(dst / p.name)
        with ThreadPoolExecutor(max_workers=8) as ex:
            list(ex.map(_f, sorted(src.glob('*'))))
        print(f'  {s}/images_{factor}: {len(list(dst.glob("*")))}')

In [ ]:
# @title 3. Restore the frozen nerfacto backbone -- per scene (NEVER trains)
# Same restore logic as the training notebook's cell 3, with the training branch
# replaced by a hard failure: this notebook only measures backbones that already
# exist. Attach them as a Kaggle Dataset (see the closing markdown cell).
import glob, subprocess, os, tarfile, re, shutil

def newest_config(s):
    # only a config whose run dir actually holds a loadable checkpoint counts --
    # this skips a half-trained run and, importantly, a stale malformed restore
    # dir (e.g. a bare `nerfacto_<scene>/` with no nerfstudio_models/), so the
    # restore branch below can rebuild it correctly.
    for pat in (f'{NERF_OUTPUT_DIR}/{s}/nerfacto/*/config.yml',
                f'{WORK}/**/{s}/nerfacto/*/config.yml'):
        c = sorted(p for p in glob.glob(pat, recursive=True)
                   if glob.glob(os.path.join(os.path.dirname(p), 'nerfstudio_models', '*.ckpt')))
        if c:
            return c[-1]
    return None

def repoint_output_dir(cfg_path, target=f'{WORK}/outputs'):
    # configs from another machine bake in an absolute output_dir; repoint it so
    # nerfstudio resolves <output_dir>/<exp>/nerfacto/<ts>/nerfstudio_models here.
    # nerfstudio serialises a Path as a multi-line !!python/object/apply:pathlib
    # .PosixPath + block-sequence-of-parts; a from-scratch config uses a plain
    # scalar. Pick the branch by which FORM is present -- not by whether re.sub
    # changed anything (a config already pointing here yields an identical sub).
    s = open(cfg_path).read()
    pathlib_pat = r'output_dir:(?: &\S+)? !!python/object/apply:pathlib\.PosixPath\n(?:- .*\n)+'
    block = ('output_dir: !!python/object/apply:pathlib.PosixPath\n'
             + ''.join(f'- {p}\n' for p in ['/'] + target.strip('/').split('/')))
    if re.search(pathlib_pat, s):
        s = re.sub(pathlib_pat, block, s, count=1)
    elif re.search(r'^output_dir: .+$', s, flags=re.M):
        s = re.sub(r'^output_dir: .+$', f'output_dir: {target}', s, count=1, flags=re.M)
    else:
        raise RuntimeError(f'no output_dir key found in {cfg_path}')
    open(cfg_path, 'w').write(s)

# Extract any attached tarballs once, up front -- each may contain one or more
# scenes' outputs/<scene>/nerfacto/<ts>/ trees; newest_config() below then finds
# whichever scenes they cover.
for tar in PRETRAINED_TARS:
    print('Extracting nerfacto tarball', tar)
    with tarfile.open(tar) as t:
        t.extractall(WORK)
for cfg in glob.glob(f'{WORK}/**/nerfacto/*/config.yml', recursive=True):
    repoint_output_dir(cfg)

NERF_CONFIGS, NERF_RESTORED = {}, {}
for s in SCENES:
    # no NERFACTO_FORCE_RETRAIN branch here: there is nothing to retrain into, so
    # clearing a run dir could only destroy the thing we came to measure.
    cfg = newest_config(s)

    if cfg is None and s in PRETRAINED_NERF:
        src_run = os.path.dirname(PRETRAINED_NERF[s])
        # nerfstudio's eval_setup recomputes the checkpoint dir from the config's
        # own output_dir/experiment_name/method_name/timestamp/relative_model_dir
        # -> the run directory MUST be named after the config's `timestamp`, and
        # the ckpt MUST sit under nerfstudio_models/. Neither holds for a packaged
        # `nerfacto_<scene>/` bundle or the repo's flat committed checkpoint, so
        # rebuild the canonical layout rather than copying the folder as-is.
        cfg_txt = open(f'{src_run}/config.yml').read()
        m = re.search(r'^timestamp:\s*(\S+)\s*$', cfg_txt, flags=re.M)
        ts = m.group(1) if m else 'restored'
        dst_run = f'{NERF_OUTPUT_DIR}/{s}/nerfacto/{ts}'
        print(s, ': restoring nerfacto from', src_run, '->', dst_run)
        os.makedirs(f'{dst_run}/nerfstudio_models', exist_ok=True)
        shutil.copy(f'{src_run}/config.yml', f'{dst_run}/config.yml')
        dpt = f'{src_run}/dataparser_transforms.json'
        if os.path.isfile(dpt):
            shutil.copy(dpt, f'{dst_run}/dataparser_transforms.json')
        ckpts = glob.glob(f'{src_run}/**/*.ckpt', recursive=True)
        assert ckpts, f'no *.ckpt found under {src_run}'
        for ck in ckpts:
            shutil.copy(ck, f'{dst_run}/nerfstudio_models/{os.path.basename(ck)}')
        repoint_output_dir(f'{dst_run}/config.yml')
        cfg = newest_config(s)

    NERF_RESTORED[s] = cfg is not None

    if cfg:
        print(s, ': using frozen nerfacto checkpoint', cfg)
    else:
        raise RuntimeError(
            f'{s}: no nerfacto checkpoint found, and this notebook never trains one -- '
            f'it only measures backbones that already exist. Attach a Kaggle Dataset '
            f'holding either .../{s}/nerfacto/<timestamp>/config.yml with '
            f'nerfstudio_models/*.ckpt, or .../nerfacto_{s}/config.yml (the '
            f'vf_nerf_outputs.zip bundle layout) -- or drop {s!r} from SCENES in cell 0. '
            f'Train it with notebooks/train_vf_nerf_kaggle.ipynb.')

    assert cfg, f'no nerfacto config produced for {s}'
    # nerfstudio loads <dir(cfg)>/nerfstudio_models/step-*.ckpt -- and the dir
    # name must equal the config's timestamp (eval_setup rebuilds the path from
    # the config, it does not trust cfg's location). Verify both here.
    run_dir = os.path.dirname(cfg)
    ck = glob.glob(f'{run_dir}/nerfstudio_models/*.ckpt')
    ts_ok = re.search(r'^timestamp:\s*(\S+)\s*$', open(cfg).read(), flags=re.M)
    ts_ok = ts_ok and ts_ok.group(1) == os.path.basename(run_dir)
    assert ck and ts_ok, (
        f'{s}: bad nerfacto layout -- ckpt={bool(ck)}, dirname matches timestamp={bool(ts_ok)} '
        f'({run_dir})')
    print(s, ': NERF_CONFIG =', cfg)
    print(s, ': checkpoint  =', ck[-1])
    NERF_CONFIGS[s] = cfg

In [ ]:
# @title 4. Restore the conditional Normalizing Flow -- per scene (NEVER trains)
# The .pt is used exactly as provided: train_conditional_nf.py has no resume flag, and
# this notebook measures flows that already exist. INPUT_COND_PT was populated by cell 2
# from whatever Kaggle Datasets are attached (it accepts both
# .../conditional_nf/<scene>/latest.pt and the vf_nerf_outputs.zip bundle layout
# .../conditional_nf_<scene>/latest.pt).
# NB: the checkpoint must match the frozen nerfacto restored above -- the flow models
# (surface point, direction) in THAT NeRF's normalised world frame, and it was fitted on
# DINO features from that scene's images_<downscale>/. Mixing a flow from one run with a
# backbone from another silently measures nonsense.
import os, shutil

COND_LATEST = {}
for s in SCENES:
    src = INPUT_COND_PT.get(s)
    if not src:
        raise RuntimeError(
            f'{s}: no conditional-NF checkpoint found, and this notebook never trains one. '
            f'Attach a Kaggle Dataset holding .../conditional_nf/{s}/latest.pt (or '
            f'cond_nf_step_*.pt), or .../conditional_nf_{s}/latest.pt from a prior run\'s '
            f'vf_nerf_outputs.zip -- or drop {s!r} from SCENES in cell 0. '
            f'Train it with notebooks/train_vf_nerf_kaggle.ipynb (cell 5).')
    os.makedirs(COND_DIR(s), exist_ok=True)
    COND_LATEST[s] = f'{COND_DIR(s)}/latest.pt'
    if os.path.abspath(src) != os.path.abspath(COND_LATEST[s]):
        shutil.copy(src, COND_LATEST[s])
    print(s, ': conditional-NF checkpoint', src)

for s in SCENES:
    print(f'--- {s} ---')
    !ls -la {COND_DIR(s)}

In [ ]:
# @title 5. Mean / median log-likelihood on both splits (per scene)
# Scores the flow's own training objective -- log_prob of (frozen-NeRF surface point, ray
# direction) conditioned on that pixel's DINO feature -- on the TRAIN frames it was fitted
# on and on nerfstudio's held-out frames, and reports the gap.
#
# NUM_BATCHES is 100 here rather than the script's default of 20: the batches themselves
# are cheap (a 4096-ray depth render plus a flow forward), while the one-time DINO cache
# per split dominates the runtime, so 5x the samples costs almost nothing and makes the
# tail statistics meaningful. The train-split cache is reused from training if present;
# the held-out split builds its own the first time (~1-2 min).
import subprocess, os, json

EVAL_DIR = f'{WORK}/eval'
os.makedirs(EVAL_DIR, exist_ok=True)
NUM_BATCHES, BATCH_SIZE = 100, 4096   # 409,600 sampled pixels per split
WORST_K = 20                          # lowest-log_prob samples dumped in full

COND_NF_LL = {}
for s in SCENES:
    out = f'{EVAL_DIR}/{s}_cond_nf_ll.json'
    cmd = [f'{VENV}/python', '-u', 'scripts/eval_cond_nf_likelihood.py',
           '--nerf-config', NERF_CONFIGS[s], '--scene-dir', SCENE_DATA(s),
           '--cond-nf-checkpoint', COND_LATEST[s], '--output-path', out,
           '--num-batches', str(NUM_BATCHES), '--batch-size', str(BATCH_SIZE),
           '--worst-k', str(WORST_K)]
    print(f'--- {s} ---'); print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO,
                   env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    COND_NF_LL[s] = json.load(open(out))
print('\nwrote:', sorted(os.listdir(EVAL_DIR)))

In [ ]:
# @title 6. Summary table, tail accounting and plots
# READ THE MEDIAN, NOT THE MEAN. log_prob is a log-DENSITY of a continuous 6-D
# distribution: it has no floor, and the target lives on a lower-dimensional manifold
# (directions are unit vectors; points are NeRF surface points), so the flow drives the
# density very high on that manifold and off a cliff just outside it. One ray whose
# frozen-NeRF depth is garbage -- a window, the unbounded background -- lands far off it
# and can carry the entire variance on its own. The tail line below says exactly how much.
import json, math
import numpy as np
import matplotlib.pyplot as plt

hdr = (f"{'scene':<10}{'split':<7}{'median':>9}{'trim .1%':>10}{'IQR':>8}"
       f"{'| raw mean':>12}{'raw std':>10}{'min':>12}{'samples':>10}")
print(hdr); print('-' * len(hdr))
for s, d in COND_NF_LL.items():
    for split, r in d['splits'].items():
        print(f"{s:<10}{split:<7}{r['median_log_prob']:>9.3f}"
              f"{r['trimmed_mean_log_prob_0.1pct']:>10.3f}{r['iqr_log_prob']:>8.3f}"
              f"{r['mean_log_prob']:>12.3f}{r['std_log_prob']:>10.2f}"
              f"{r['min_log_prob']:>12.1f}{r['n_samples']:>10d}")
    gaps = [(k, d[k]) for k in ('median_train_minus_test', 'trimmed_train_minus_test',
                                'train_minus_test') if k in d]
    print(f"{'':<10}{'gap':<7}" + '   '.join(f'{k.replace("_train_minus_test", "")}'
                                             f' {v:+.3f}' for k, v in gaps))

print('\ntail accounting -- how much of the variance a handful of samples own:')
for s, d in COND_NF_LL.items():
    for split, r in d['splits'].items():
        t = r['tail_contribution']['lowest_0.001pct']
        u = r['tail_contribution']['lowest_0.1pct']
        print(f"  {s:<9}{split:<6} lowest {t['n']:>3} (<= {t['threshold']:>10.1f}): "
              f"{100 * t['variance_share']:>5.1f}% of var, {t['mean_shift']:+.3f} of mean"
              f"   | lowest 0.1% ({u['n']}): {100 * u['variance_share']:>5.1f}% of var")

print('\nis it a bad NeRF depth?  (Spearman rank correlation, per split)')
for s, d in COND_NF_LL.items():
    for split, r in d['splits'].items():
        print(f"  {s:<9}{split:<6} log_prob vs depth {r['spearman_log_prob_vs_depth']:+.3f}"
              f"   vs |P| {r['spearman_log_prob_vs_point_norm']:+.3f}"
              f"   depth p0={r['depth_stats']['p0']:.3f} "
              f"p50={r['depth_stats']['p50']:.3f} p100={r['depth_stats']['p100']:.3f}")

# --- histograms: x-clipped to the central 99.8% so one blow-up cannot flatten every bin
fig, axes = plt.subplots(1, len(COND_NF_LL), figsize=(5 * len(COND_NF_LL), 3.6),
                         squeeze=False)
for ax, (s, d) in zip(axes[0], COND_NF_LL.items()):
    for split, r in d['splits'].items():
        h = r['histogram']
        edges, counts = np.array(h['bin_edges']), np.array(h['counts'])
        centres = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centres, counts / counts.sum(), lw=1.4,
                label=f"{split} (median {r['median_log_prob']:.2f}, "
                      f"{h['underflow']} below axis)")
    ax.set_title(s); ax.set_xlabel('log_prob (central 99.8%)')
    ax.set_ylabel('fraction of samples'); ax.legend(fontsize=8)
fig.suptitle('Per-sample log-likelihood -- the bulk, with the tail clipped off the axis')
fig.tight_layout(); plt.show()

# --- per-image means: does one frame own the damage?
fig, axes = plt.subplots(1, len(COND_NF_LL), figsize=(5 * len(COND_NF_LL), 3.4),
                         squeeze=False)
for ax, (s, d) in zip(axes[0], COND_NF_LL.items()):
    for split, r in d['splits'].items():
        vals = [im['median'] for im in r['per_image']]
        ax.plot(range(len(vals)), vals, marker='.', lw=0.7, label=f'{split} (n={len(vals)})')
    ax.set_title(s); ax.set_xlabel('image index within split')
    ax.set_ylabel('median log_prob'); ax.legend(fontsize=8)
fig.suptitle('Per-image median -- a frame the flow generalises badly to shows up here')
fig.tight_layout(); plt.show()

# --- the worst samples themselves, with the depth that produced them
for s, d in COND_NF_LL.items():
    for split, r in d['splits'].items():
        w = r['worst_samples'][:8]
        if not w:
            continue
        print(f"\nworst {len(w)} samples -- {s} / {split}   "
              f"(split median depth {r['depth_stats']['p50']:.3f})")
        print(f"  {'log_prob':>12}{'depth':>10}{'|P|':>9}  image")
        for e in w:
            print(f"  {e['log_prob']:>12.1f}{e['depth']:>10.3f}{e['point_norm']:>9.3f}"
                  f"  {e['image']}")

## Reading the numbers

**The gap.** `median_train_minus_test` is the one to quote. Positive is the usual
direction (the flow prefers what it was fitted on); around zero means it generalises to
unseen viewpoints; negative means the splits are crossed and something is wrong with the
restore. Compare it with `train_minus_test` (the mean-based original): if the two differ
much, the mean is being dragged by its tail, and the tail accounting below the table says
by how much.

**Absolute values are not comparable across scenes.** A log-density is only defined up to
the units of the space it lives in, and each scene has its own `dataparser_scale`. Compare
train-vs-test *within* a scene, and compare gaps between scenes -- never the raw levels.

**The tail line.** `lowest 1 (<= -13401.2): 99.9% of var, -0.164 of mean` means exactly
what it says: drop that one sample and the variance essentially vanishes. That is a
property of the frozen NeRF's depth on a handful of rays, not of the flow's fit. If the
lowest 0.1% carries most of the variance instead, the damage is spread out and worth
chasing.

**Is it the depth?** `spearman_log_prob_vs_depth` correlates every sample's log-prob with
the depth its ray rendered. Strongly non-zero, plus worst-sample depths far from the split
median, is the bad-depth story confirmed -- those rays hit the unbounded background where
nerfacto's depth is meaningless, so `P` lands nowhere near any surface the flow ever saw.
Near zero means look elsewhere: an under-trained flow, or a DINO cache that does not match
the images the backbone was trained on.

**Per-image plot.** A single frame dipping well below the rest is a viewpoint the flow
generalises badly to. In the held-out split that is the interesting case; in the train
split it suggests that frame was under-sampled or its DINO features are unusual.

**`n_nonfinite`** should be 0. Anything else means `log_prob` returned NaN/inf and those
samples were excluded from every statistic -- worth investigating before trusting the run.

## Attaching checkpoints

Same mechanism as the training notebook. **kaggle.com -> Create -> New Dataset**, upload
either a prior run's `vf_nerf_outputs.zip` (already in the bundle layout
`nerfacto_<scene>/`, `conditional_nf_<scene>/` -- do not rename or flatten those folders)
or the loose trees:

- nerfacto: `.../<scene>/nerfacto/<timestamp>/config.yml` with
  `nerfstudio_models/*.ckpt`, **or** `.../nerfacto_<scene>/config.yml`;
- conditional-NF: `.../conditional_nf/<scene>/latest.pt` (or `cond_nf_step_*.pt`),
  **or** `.../conditional_nf_<scene>/latest.pt`.

Then right sidebar -> **Input -> Add Input** -> your dataset -> **Add**, and run from
cell 1. Trim `SCENES` in cell 0 to the scenes you actually attached.

Outputs land in `/kaggle/working/eval/` as `<scene>_cond_nf_ll.json` plus the
`<scene>_cond_nf_ll_samples.npz` sidecar.